# ETL aditivo: dataset extendido

Ejecuta `src/etl_features_ext.py` para generar `data/processed/airprice_features_ext.csv`.

**Estrategia:** el dataset limpio se derivó de `data/listings.csv`, así que se recuperan columnas del original con un merge uno-a-uno por `id` (no se reprocesa todo el CSV). El dataset limpio y `airprice_features.csv` **no se modifican**.

**Qué se agrega:**

- Variables del original: `host_listings_count`, `host_total_listings_count`, `review_scores_*` (6 subpuntuaciones), `availability_30/60/90`, `number_of_reviews_ltm/l30d/ly`, `minimum/maximum_nights_avg_ntm`, `calculated_host_listings_count_*`, `host_since`, `first_review`, `last_review`, `amenities`.
- Derivadas: antigüedad del host, días desde reviews, ratios de disponibilidad y reviews, ratios de portafolio del host, rangos de noches, ratios de capacidad.
- Amenidades: ~50 flags binarias (`amenity_wifi`, `amenity_elevator`, `amenity_pool`, etc.).
- Geo adicional: distancia mínima a POIs, a estaciones, a landmarks y a aeropuertos; número de POIs a 2 km; densidad de anuncios a 1 km, distancia al vecino más cercano, densidad por vecindario/cluster; K-Means con k=5, 8 y 10 ajustado solo con train.

**Fuga de datos:** se excluyen `price`, `price_quote_*`, `estimated_revenue_l365d` y `estimated_occupancy_l365d`.

In [2]:
import sys
sys.path.append('../..')

import pandas as pd

from src.etl_features_ext import (
    AUDIT_PATH,
    METADATA_PATH,
    SALIDA_PATH,
    construir_dataset_ext,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

In [1]:
print(2+2)

4


In [3]:
df = construir_dataset_ext()
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
print('Guardado en:', SALIDA_PATH)
print('Auditoría:', AUDIT_PATH)
print('Metadatos:', METADATA_PATH)

Filas: 61617
Columnas: 166
Guardado en: D:\Universidad\ElectivaCienciasComputacion2\Proyecto\AirPrice\data\processed\airprice_features_ext.csv
Auditoría: D:\Universidad\ElectivaCienciasComputacion2\Proyecto\AirPrice\data\processed\airprice_features_ext_audit.csv
Metadatos: D:\Universidad\ElectivaCienciasComputacion2\Proyecto\AirPrice\data\processed\airprice_features_ext_metadata.json


In [4]:
auditoria = pd.read_csv(AUDIT_PATH)
display(auditoria.T)

print('Tasa de emparejamiento por id:', float(auditoria['tasa_emparejamiento_id'].iloc[0]))
print('Columnas nuevas:', int(auditoria['columnas_nuevas'].iloc[0]))

,0
filas_base,61617
filas_ext,61617
columnas_base,38
columnas_ext,166
columnas_nuevas,128
tasa_emparejamiento_id,0.998929
columnas_originales_recuperadas,id;last_scraped;host_since;host_response_time;...
columnas_excluidas_por_fuga,price;price_quote_checkin_date;price_quote_che...
kmeans_k,5;8;10
n_amenities_flags,49


Tasa de emparejamiento por id: 0.9989288670334484
Columnas nuevas: 128


In [5]:
columnas_geo = [
    c for c in df.columns
    if c.startswith('distancia_') or c.startswith('cluster_zona') or c in (
        'vecinos_1km', 'pois_2km', 'densidad_vecindario', 'ratio_densidad_1km'
    )
]
print('Columnas geo extendidas:', columnas_geo)
print('Flags de amenidades:', [c for c in df.columns if c.startswith('amenity_')][:10], '...')

df[['id', 'price_range'] + columnas_geo].head()

Columnas geo extendidas: ['distancia_centro_km', 'distancia_heathrow_km', 'distancia_kings_cross_km', 'distancia_hyde_park_km', 'distancia_canary_wharf_km', 'cluster_zona', 'distancia_min_poi_km', 'distancia_metro_km', 'distancia_landmark_km', 'distancia_aeropuerto_km', 'pois_2km', 'vecinos_1km', 'distancia_vecino_km', 'densidad_vecindario', 'ratio_densidad_1km', 'cluster_zona_k5', 'cluster_zona_k8', 'cluster_zona_k10']
Flags de amenidades: ['amenity_wifi', 'amenity_kitchen', 'amenity_air_conditioning', 'amenity_heating', 'amenity_washer', 'amenity_dryer', 'amenity_elevator', 'amenity_gym', 'amenity_pool', 'amenity_hot_tub'] ...


,id,price_range,distancia_centro_km,distancia_heathrow_km,distancia_kings_cross_km,distancia_hyde_park_km,distancia_canary_wharf_km,cluster_zona,distancia_min_poi_km,distancia_metro_km,distancia_landmark_km,distancia_aeropuerto_km,pois_2km,vecinos_1km,distancia_vecino_km,densidad_vecindario,ratio_densidad_1km,cluster_zona_k5,cluster_zona_k8,cluster_zona_k10
0,11551,ALTO,5.213260,23.347245,7.778892,6.137188,8.177426,1,0.142587,0.142587,3.444827,12.555169,2,441,0.007495,2930,1.040094,2,2,4
1,13913,MEDIO,6.885899,26.055211,4.273762,7.740364,9.352206,0,0.690819,0.690819,3.583660,13.273994,1,354,0.015936,2693,0.834906,0,6,8
2,15400,MEDIO,3.541761,19.916570,5.680997,2.174816,10.201576,1,0.034552,1.856043,0.034552,15.183795,9,972,0.016431,4513,2.292453,3,5,9
3,17402,PREMIUM,1.855887,22.448771,1.540913,2.364154,8.332567,1,0.775654,0.775654,1.005311,13.315734,15,1475,0.016336,8076,3.478774,0,2,4
4,513198,ECONOMICO,2.435882,23.979882,4.781635,4.413137,6.224001,0,1.617867,1.617867,1.697555,11.139646,3,788,0.039795,2930,1.858491,0,2,4


In [6]:
nulos = (
    df.isna().mean()
    .sort_values(ascending=False)
    .head(20)
    .rename('proporcion_nulos')
    .to_frame()
)
display(nulos)

print('Duplicados de id:', int(df['id'].duplicated().sum()))
print('price_clean en el dataset extendido:', 'price_clean' in df.columns)
print('price en el dataset extendido:', 'price' in df.columns)
print('estimated_revenue_l365d en el dataset extendido:', 'estimated_revenue_l365d' in df.columns)

,proporcion_nulos
antiguedad_host_anios,1.000000
neighbourhood,1.000000
host_total_listings_count,1.000000
host_verifications,1.000000
host_response_time,1.000000
host_since,1.000000
host_listings_ratio,1.000000
review_scores_checkin,0.215833
review_scores_location,0.215833
review_scores_value,0.215817


Duplicados de id: 0
price_clean en el dataset extendido: True
price en el dataset extendido: False
estimated_revenue_l365d en el dataset extendido: False


## Notas de implementación

- El merge conserva el orden de `airprice_features.csv`, por lo que el split estratificado es idéntico al de los escenarios base y geo.
- Los K-Means se ajustan **solo con las coordenadas de train**; luego se asignan todos los registros. Así se evita usar información del test para construir Features.
- La densidad (`vecinos_1km`, `distancia_vecino_km`) se calcula con `BallTree` y distancia haversine. Es una propiedad del territorio, no del target; no hay fuga.
- Algunas variables del plan original no existen en esta versión del scrape (90 columnas): `guests_included`, `extra_people`, `security_deposit`, `cleaning_fee` no están disponibles. No se inventan.
- `estimated_occupancy_l365d` se excluye por ser una estimación derivada del historial, no una característica observable al publicar el anuncio.